# Lab 24 — Chạy TOÀN BỘ lab trên Google Colab (GPU)

## Vì sao phải lên Colab

Pipeline nạp **2 model 568M tham số** vào cùng 1 process:

| Nơi nạp | Model | fp32 |
|---|---|---|
| `src/m2_search.py` → `DenseSearch` | `BAAI/bge-m3` | ~2.3 GB |
| `src/m3_rerank.py` → `CrossEncoderReranker` | `BAAI/bge-reranker-v2-m3` | ~2.3 GB |

`setup_answers.py` nạp **cả hai cùng lúc** (reranker trước, encoder sau) → ~5–6 GB RSS + torch runtime,
chưa kể mmap safetensors nhân đôi lúc load. Trên Windows 8–16 GB có OneDrive + VS Code → tràn RAM.

Trên Colab T4: **model nằm ở 15 GB VRAM**, RAM hệ thống chỉ còn ~3 GB → hết tràn.
`SentenceTransformer` / `CrossEncoder` tự bắt CUDA nên **không cần sửa một dòng code nào**.

Qdrant cũng không cần Docker: `m2_search.py` đã có sẵn fallback embedded (`QdrantClient(path=...)`).

## Chuẩn bị runtime (làm ĐÚNG thứ tự)

1. **Runtime → Disconnect and delete runtime** (bắt buộc nếu runtime này đã từng cài package).
2. **Runtime → Change runtime type** → Runtime version **2025.07 (Python 3.11)** + Hardware **T4 GPU**.
   Không dùng Python 3.13 — RAGAS 0.1.22 khoá LangChain < 0.3.
3. Thêm 2 Colab Secrets (🔑 thanh bên trái) và bật *Notebook access*: `OPENAI_API_KEY`, `DEEPSEEK_API_KEY`.


## 1 · Kiểm tra phần cứng

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | head -2
import sys; print('Python:', sys.version.split()[0])


## 2 · Lấy code về

> ⚠️ Trên GitHub hiện mới chỉ có commit scaffold. `src/m1..m5`, `src/pipeline.py`, `answers_50q.json`
> và các sửa đổi trong `src/phase_*.py` **chưa được push** → clone về sẽ là code cũ còn TODO.
>
> Ở máy local chạy trước: `git add -A && git commit -m "wip" && git push` (`.env` đã nằm trong `.gitignore`,
> không bị lộ key). Hoặc dùng đường upload zip ở cuối cell.


In [ ]:
import os

REPO = 'Day24-Track3-Production-Eval-Guardrail-Stack-2A202601618-NguyenDucAnhTuan'
DEST = f'/content/{REPO}'
# Dùng {REPO} chứ KHÔNG dùng $REPO: IPython hiểu `$REPO.git` là truy cập thuộc
# tính `.git` của biến, expand thành rỗng -> clone vào thẳng /content và fail.
if not os.path.isdir(f'{DEST}/.git'):
    !git clone https://github.com/nt15032/{REPO}.git {DEST}
else:
    !git -C {DEST} pull --ff-only
os.chdir(DEST)
assert os.path.isdir('src'), f'Clone hỏng — {DEST} không có src/'
print('cwd:', os.getcwd())
!ls src/

# Repo private, hoặc muốn dùng code local đang sửa dở?
#   -> nén thư mục ở máy thành lab24.zip rồi:
#   from google.colab import files; files.upload()
#   !unzip -q lab24.zip -d /content


## 3 · Cài dependencies (pin cứng — chạy đúng một lần)

Các version dưới đây lấy từ `.venv` ở máy đã chạy được cả 3 phase, không phải đoán.
Bốn thứ **bắt buộc** giữ nguyên: `numpy==1.26.4` (ragas/pyarrow), `langchain-core==0.2.43`
(ragas 0.1.22 khoá LangChain < 0.3), `nemoguardrails==0.23.0` (output rail `self check output`
trong `guardrails/config.yml` cần API `GenerationOptions(rails=[...])`), và `thinc==8.3.13`
— thinc bản Colab cài sẵn (8.3.6) đòi numpy ≥ 2, sẽ làm spaCy/Presidio gãy ở Phase C.

pip sẽ in cảnh báo `opencv-python-headless ... requires numpy>=2`: bỏ qua được,
notebook này không dùng opencv.


In [ ]:
# numpy pin nằm CHUNG một lệnh với phần còn lại: tách ra 2 lệnh thì lệnh sau
# có thể kéo numpy 2.x về lại và cell sanity check sẽ assert fail.
%pip install -q --no-cache-dir --force-reinstall "numpy==1.26.4"
%pip install -q --no-cache-dir \
  "numpy==1.26.4" "pytest==8.3.4" \
  "ragas==0.1.22" "datasets==5.0.1" "pandas==2.3.3" "pyarrow==25.0.1" "scipy==1.17.1" \
  "langchain==0.2.17" "langchain-core==0.2.43" "langchain-community==0.2.19" "langchain-openai==0.1.25" \
  "sentence-transformers==6.0.0" "transformers==5.15.1" "openai==1.109.1" \
  "qdrant-client==1.12.1" "rank-bm25==0.2.2" "underthesea==9.5.0" "pypdf==5.1.0" "python-dotenv==1.0.1" \
  "nemoguardrails==0.23.0" "presidio-analyzer==2.2.355" "presidio-anonymizer==2.2.355" \
  "spacy==3.8.16" "thinc==8.3.13"


> ### ⚠️ BẮT BUỘC: **Runtime → Restart session** ngay bây giờ
>
> NumPy vừa bị hạ từ 2.x xuống 1.26.4, process hiện tại vẫn giữ bản cũ trong bộ nhớ.
>
> Sau khi kết nối lại: **KHÔNG chạy lại 2 cell pip ở trên**. Bắt đầu từ cell dưới đây.


## 4 · Sanity check sau restart

In [ ]:
import os, sys

os.chdir('/content/Day24-Track3-Production-Eval-Guardrail-Stack-2A202601618-NguyenDucAnhTuan')
print('cwd:', os.getcwd())
import numpy as np, torch
assert sys.version_info[:2] == (3, 11), f'Cần runtime 2025.07 / Python 3.11, đang là {sys.version}'
assert np.__version__ == '1.26.4', f'NumPy sai ({np.__version__}) — cài lại rồi Restart session đúng 1 lần'
assert torch.cuda.is_available(), (
    'Không có GPU → 2 model 2.3GB sẽ nằm trên RAM, lặp lại đúng lỗi tràn RAM ở local. '
    'Runtime → Change runtime type → T4 GPU.')
print(f'OK — numpy {np.__version__}, torch {torch.__version__}, GPU {torch.cuda.get_device_name(0)}')


## 5 · API keys → ghi ra `.env` (config.py đọc file này)

In [ ]:
from google.colab import userdata
from pathlib import Path

keys = {k: userdata.get(k) for k in ('OPENAI_API_KEY', 'DEEPSEEK_API_KEY')}
missing = [k for k, v in keys.items() if not (v and v.strip())]
assert not missing, f'Thiếu Colab Secret: {missing} (🔑 thanh bên trái, nhớ bật Notebook access)'

Path('.env').write_text(
    f"OPENAI_API_KEY={keys['OPENAI_API_KEY']}\n"
    f"DEEPSEEK_API_KEY={keys['DEEPSEEK_API_KEY']}\n"
    'DEEPSEEK_BASE_URL=https://api.deepseek.com\n'
    'LLM_MODEL=deepseek-chat\n', encoding='utf-8')
print('.env đã ghi (không in giá trị key)')


## 6 · spaCy model cho Presidio (~590MB, ~1 phút)

In [ ]:
!python -m spacy download en_core_web_lg 2>&1 | tail -3


## 7 · Setup — sinh `answers_50q.json`

Đây chính là bước tràn RAM ở local: chunk → M5 enrich (100 API call) → index BM25+dense → rerank → sinh 50 answer.
Khoảng 5–8 phút trên T4. Cell in `nvidia-smi` + `free` ở cuối: model phải nằm ở **VRAM**, RAM còn dư nhiều.


In [ ]:
!python setup_answers.py
!nvidia-smi --query-gpu=memory.used --format=csv,noheader; free -g | head -2


## 8 · Phase A — RAGAS 50q

In [ ]:
!python src/phase_a_ragas.py

import json
report = json.load(open('reports/ragas_50q.json', encoding='utf-8'))
print('\nevaluation_mode:', report['evaluation_mode'], '<- phải là ragas_deepseek, KHÔNG phải offline_proxy')
for dist, stats in report['per_distribution'].items():
    print(f"  {dist:<12} avg={stats['avg_score']:.3f}  faithfulness={stats['faithfulness']:.3f}")
print('bonus +4 nếu adversarial avg < factual avg')


## 9 · Phase B — LLM-as-Judge (gpt-4o-mini)

In [ ]:
!python src/phase_b_judge.py

import json
report = json.load(open('reports/judge_results.json', encoding='utf-8'))
print('\njudge_model:', report['judge_model'], '<- phải là gpt-4o-mini')
print('cohen_kappa:', report['cohen_kappa'], '(bonus +3 nếu > 0.6)')


## 10 · Phase C — Presidio + NeMo Guardrails

In [ ]:
!python src/phase_c_guard.py

import json
report = json.load(open('reports/guard_results.json', encoding='utf-8'))
print(f"\npass_rate: {report['passed']}/{report['total']} (bonus +3 nếu >= 18/20)")
print('latency P95 total:', report['latency']['total_ms']['p95'], 'ms')


## 11 · Check trước khi nộp

In [ ]:
!pytest tests/ -q
!python check_lab.py


## 12 · Lấy kết quả về máy

Nhớ mở lại `reports/blueprint.md` và điền **P95 latency thực tế**, **adversarial pass rate**,
**worst RAGAS metric**, **dominant failure distribution** đúng theo số vừa đo ở cell 8–10
(Task 13 chấm riêng 10 điểm).


In [ ]:
!zip -qr lab24_outputs.zip answers_50q.json reports/
from google.colab import files
files.download('lab24_outputs.zip')


---

## Xử lý sự cố

| Triệu chứng | Nguyên nhân | Cách xử lý |
|---|---|---|
| `evaluation_mode: offline_proxy` | `evaluate_ragas()` nuốt exception rồi rơi về fallback token-overlap | Chạy lại cell 8, đọc dòng `⚠️ RAGAS evaluation failed:` để biết lỗi thật |
| `judge_model: offline_deterministic_fallback` | `OPENAI_API_KEY` rỗng hoặc hết quota | Kiểm tra lại cell 5 |
| `numpy.dtype size changed` | Quên Restart session sau khi pip | Restart session rồi chạy lại từ cell 4 |
| Kernel crash ở `setup_answers.py` | Runtime CPU chứ không phải GPU | Cell 4 đã assert chặn trước — đổi sang T4 |
| `Storage folder qdrant_local_data is already accessed` | Process cũ còn giữ lock | `!rm -rf qdrant_local_data` rồi chạy lại |
| Colab ngắt kết nối giữa chừng | Idle timeout | `answers_50q.json` đã lưu → chạy lại từ cell 8, không cần lặp cell 7 |

## Chạy lại từng phase

3 phase độc lập, đều chỉ đọc `answers_50q.json`. Cell 7 (tốn ~8 phút + 100 API call) chỉ cần chạy **một lần**;
sửa `src/phase_*.py` xong thì chỉ chạy lại đúng cell của phase đó.
